# 02 — Verify a two-layer extension, then test an effective pump

**Learning goals:** verify a conservative reservoir extension; explain a
maintained gradient as pump–mixing balance; implement and inventory-check a
finite synthetic carbon input.

Part A is a structural check. Part B uses the **calibration-first** route.
Part C tests forcing mechanics and analytical consistency. Distinct-pump
attribution, OA/OAE interpretation, sediments and feedbacks belong in 03/04.

**Core time: 55 minutes.** A: reservoir and mixing (20); B: derive and implement
the effective pump (20); C: derive the addition and run supplied forcing (15).
Fill the named quantities inside the marked blocks; constructor syntax,
chemistry, plotting, restarts and audits are supplied. Predict, map, run, check,
then explain. [Teaching goals](../../TEACHING_GOALS.md).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import PyCO2SYS as pyco2

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'teaching_config.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from teaching_config import TEACHING as C
from simple_models import (
    new_model, box_parameters, box_mass_kg, connect_atmosphere,
    single_box, inventories, audit, mixing_mass_transport,
    integrated_signal,
)
from esbmtk import (
    GasReservoir, Species2Species, Signal, Source,
    initialize_reservoirs, create_bulk_connections, add_carbonate_system_1,
)
from model import run_model
from teaching_plots import plot_pump_comparison, plot_synthetic_forcing

## A. Add a reservoir without changing the equilibrium problem

```text
Atmosphere (xCO2) ↔ Surface (DIC_s, TA_s) ↔ Deep (DIC_d, TA_d)
                   gas exchange           mixing: DIC and TA
                                          pump added only in Part B
```

Only the surface exchanges CO2 with the atmosphere. The instructor has prepared
the layer split in `teaching_config.py` so that the reference surface/deep DIC
values (2040 and 2250 µmol/kg) give an ocean/atmosphere carbon inventory ratio
of **62.4** at 280 ppm. The surface depth is about **298.75 m** with the default
settings. It is a ratio-derived teaching geometry, not an observed mixed-layer
depth. It is calculated before running, using the full ocean volume, atmospheric
mole inventory and ESBMTK density; no rounded baseline ratio is used.

Surface and deep volumes sum to the 01 ocean volume. Both layers retain the
buffered 01 TA, total carbon, T = 16 °C, S = 35, P = 0 bar and chemistry. The
layer split changes the geometry of transport but does not replace the initial
carbon inventory. Real thermal and TA gradients are omitted here; 03/04 use
benchmark box conditions.

**Exercise:** create `Surface` and `Deep` using the supplied geometry helper,
then add equal up/down transports carrying both DIC and TA. The common initial
DIC of 1000 µmol/kg is the alternative partition checked in 01.

In [ ]:
inferred_ta = float(C.reference_state()['alkalinity'])
print('Ratio-derived surface depth (m):', C.surface_depth_m)
print('Volumes (m3):', C.surface_volume_m3, C.deep_volume_m3)
assert C.surface_volume_m3 + C.deep_volume_m3 == C.ocean_volume_m3

def build_layers(k_kg_yr=0.0, state=None):
    M = new_model()
    if state is None:
        state = {'Surface': (1000.0, inferred_ta), 'Deep': (1000.0, inferred_ta)}
    # Exercise 02.1: assign deep_volume_m3 and the two deep initial concentrations.
    # Geometry is supplied in C; state stores (DIC, TA) in umol/kg for each box.
    raise NotImplementedError("Exercise: replace this line with your solution")
    initialize_reservoirs(M, {
        'Surface': box_parameters(M, C.surface_volume_m3, *state['Surface']),
        'Deep': box_parameters(M, deep_volume_m3, deep_dic_umol_kg, deep_ta_umol_kg),
    })
    add_carbonate_system_1([M.Surface, M.Deep])
    # Supplied: Q in m3/yr times ESBMTK rho in kg/m3 gives kg/yr.
    transport_kg_yr = mixing_mass_transport()
    # Exercise 02.2: name the two directed arrows and select transported_species.
    # Names follow 'Source_to_Sink@id'; use IDs mix_down and mix_up.
    raise NotImplementedError("Exercise: replace this line with your solution")
    create_bulk_connections({
        down_arrow: {'ty': 'scale_with_concentration', 'sc': transport_kg_yr,
                     'sp': transported_species},
        up_arrow: {'ty': 'scale_with_concentration', 'sc': transport_kg_yr,
                   'sp': transported_species},
    }, M)
    if k_kg_yr:
        # Exercise 02.3 (Part B): choose pump_source, pump_sink and pump_scale.
        # This arrow transfers DIC only; its scale is the fitted kg/yr coefficient.
        # Leave this block for Part B: it is not used by the no-pump run.
        raise NotImplementedError("Exercise: replace this line with your solution")
        Species2Species(source=pump_source, sink=pump_sink,
                        ctype='scale_with_concentration', scale=float(pump_scale),
                        id='effective_pump')
    connect_atmosphere(M, [M.Surface, M.Deep])
    return M

# Supplied structural checks: compare with 01 before introducing the pump.
pump_off = build_layers()
one_box = single_box(ta_umol_kg=inferred_ta, initial_dic_umol_kg=1000)
for case in (one_box, pump_off):
    run_model(case)
    print(audit(case))
np.testing.assert_allclose(sum(box_mass_kg(b) for b in pump_off.ocean_boxes),
                           box_mass_kg(one_box.Ocean), rtol=1e-12)
np.testing.assert_allclose(pump_off.Surface.DIC.c[-1], pump_off.Deep.DIC.c[-1], atol=0.2e-6)
np.testing.assert_allclose(pump_off.Surface.DIC.c[-1], one_box.Ocean.DIC.c[-1], atol=0.2e-6)
np.testing.assert_allclose(pump_off.CO2_At.c[-1], one_box.CO2_At.c[-1], atol=0.5e-6)

### Explain the structural check

Why can symmetric mixing change the transient but not maintain a stationary
DIC gradient? What would non-conservation tell you about your connections?

> **Your explanation:** replace this placeholder with your answer.

## B. Derive the first-order pump coefficient

**Assumption:** effective downward export is first order in surface DIC:

$$J_{pump}(t)=kDIC_s(t).$$

This is an aggregate export/remineralization closure. It carries DIC only,
without water or TA. The coefficient $k$ is constant in this exercise; real
biological export need not scale with the entire DIC pool.

Use DIC in mol/kg, $Q$ in m3/yr and $\rho$ in kg/m3. Mixing returns carbon upward:

$$J_{mix}(t)=Q\rho[DIC_d(t)-DIC_s(t)].$$

The inventories obey

$$m_s\frac{dDIC_s(t)}{dt}=J_{gas}(t)+J_{mix}(t)-J_{pump}(t),$$
$$m_d\frac{dDIC_d(t)}{dt}=-J_{mix}(t)+J_{pump}(t).$$

Atmospheric tendency is $-J_{gas}(t)$. Adding all three cancels the internal
carbon fluxes. Only mixing carries TA, so the summed TA inventory is constant.

### Exercise: derive the expression before coding

1. At a stationary state, what must the deep-box tendency be? Use that condition
   to relate pump and mixing fluxes, writing stationary concentrations with stars.
2. Rearrange that relationship to obtain an expression for $k$ in terms of
   $Q$, $\rho$, $DIC_s^*$ and $DIC_d^*$. Determine the units of $k$; is it simply
   an inverse-time rate constant when DIC is expressed in mol/kg?
3. Evaluate your expression with the observed reference DIC values 2040 and
   2250 µmol/kg and the independently selected transport of 20 Sv. Which
   combination of $k$, $Q$ and $\rho$ does the DIC ratio constrain?
4. Complete Exercise 02.3 in `build_layers` and rerun its definition cell, then compare pump on
   and pump off at the same initial total carbon and TA. Report the implied
   reference export flux as well as the realized stationary export.

**Calibration-first:** the observed reference ratio is used to infer $k$.
Agreement with that ratio checks implementation; it is not an independent
prediction of the gradient. No independent export estimate is supplied here.

> **Your explanation:** replace this placeholder with your answer.

In [ ]:
raise NotImplementedError("Exercise: replace this line with your solution")
print('Fitted k (kg/yr):', k)
print('Implied reference export (Tmol/yr):', k * C.target_dic_umol_kg * 1e-6 / 1e12)
pump_on = build_layers(k)
run_model(pump_on)
print(audit(pump_on))
for label, case in [('pump off', pump_off), ('pump on', pump_on)]:
    print(label, 'xCO2 (ppm), surface/deep DIC (umol/kg):',
          case.CO2_At.c[-1] * 1e6, case.Surface.DIC.c[-1] * 1e6,
          case.Deep.DIC.c[-1] * 1e6)
np.testing.assert_allclose(pump_on.Deep.DIC.c[-1] / pump_on.Surface.DIC.c[-1],
                           C.target_deep_dic_umol_kg / C.target_dic_umol_kg, rtol=1e-4)
Jmix = mixing_mass_transport() * (pump_on.Deep.DIC.c - pump_on.Surface.DIC.c)
Jpump = k * pump_on.Surface.DIC.c
np.testing.assert_allclose(Jmix[-1], Jpump[-1], rtol=1e-4)
plot_pump_comparison(pump_off, pump_on, Jmix, Jpump)
deep_transfer = box_mass_kg(pump_on.Deep) * (pump_on.Deep.DIC.c[-1] - pump_off.Deep.DIC.c[-1])
print('Matched change in deep carbon (Pmol):', deep_transfer / 1e15)
print('Actual stationary export (Tmol/yr):', Jpump[-1] / 1e12)

### Interpret the matched experiment

Which result was fitted? Which outputs are conditional model results? Identify
the early gas-exchange adjustment and the slower approach of mixing and pump
fluxes to equality. Why need the final atmosphere not be 280 ppm?

> **Your explanation:** replace this placeholder with your answer.

## C. Calculate a finite carbon input, then verify the forcing code

The prepared geometry makes the **reference pumped ocean/atmosphere inventory
ratio 62.4** at 280 ppm and the reference DIC values. This number is an inventory
ratio, not seawater equilibrium capacity $F$. The initial total carbon still
comes from the mass-based buffered 01 configuration.

### Exercise: calculate the required addition

Before running any forcing, use the following inputs to calculate how much
extra carbon would make that reference pumped state possible:

| Input | Where to find it |
| --- | --- |
| Pumped ocean/atmosphere carbon ratio, 62.4 | `C.pumped_ocean_atmosphere_ratio` |
| Reference atmospheric xCO2, 280 ppm | `C.target_xco2_ppm` |
| Atmospheric mole inventory | `C.atmosphere_mol` |
| Existing atmosphere-plus-ocean carbon | `C.total_carbon_mol` |

1. Convert the reference atmospheric mole fraction into moles of atmospheric carbon.
2. Use 62.4 to calculate the ocean carbon and then the combined target inventory.
3. Subtract the existing total carbon to find the extra carbon to inject. Use
   the actual inventory; do not substitute a rounded baseline ratio of 57.
4. Check your answer against the extra carbon held in the deep box relative to
   a uniform ocean at the reference surface DIC. Explain why the two expressions agree.
5. Assign your result in **mol C** to `extra_carbon_mol` in the calculation cell.
   The supplied `Signal` example below will use this variable as its `mass`.

Predict the endpoint before integrating. The final return to the reference
state is a forcing-implementation and conservation check. The chosen geometry,
inferred TA and fitted pump already make that reference state consistent; the
run is not independent evidence for the pump mechanism.

> **Your explanation:** replace this placeholder with your answer.

In [ ]:
# Exercise 02.4: evaluate your derived carbon addition in mol C.
# Assign C_atm_280, target_total_carbon and extra_carbon_mol.
raise NotImplementedError("Exercise: replace this line with your solution")
print('Carbon to inject (Pmol C):', extra_carbon_mol / 1e15)

### Use the supplied forcing example

The pulse starts at 1000 yr and lasts 1000 yr. The code checks both the sampled
mass and the continuous piecewise-linear input seen by the solver, then compares
the forced run with a matching unforced restart. Keep the pulse zero at both
model boundaries, and check the budget throughout the trajectory.


In [ ]:
# Supplied forcing example: insert your calculated extra_carbon_mol above.
state = {b.name: (b.DIC.c[-1] * 1e6, b.TA.c[-1] * 1e6) for b in pump_on.ocean_boxes}
forced = build_layers(k, state=state)
control = build_layers(k, state=state)
# Recompute atmospheric carbon by conservation from the pump-on restart.
np.testing.assert_allclose(forced.CO2_At.c[0], pump_on.CO2_At.c[-1], atol=1e-10)

signal = Signal(name='synthetic_carbon', species=forced.CO2, register=forced,
                start='1000 yr', duration='1000 yr',
                mass=f'{extra_carbon_mol} mol', shape='square')
source = Source(name='external_carbon', species=forced.CO2)
connection = Species2Species(source=source, sink=forced.CO2_At,
                            rate='0 mol/yr', signal=signal, id='synthetic_input')
signal_time, signal_flux = forced.time.copy(), signal.m.copy()
sampled_mass = signal_flux.sum() * forced.dt
continuous_mass = integrated_signal(signal_time, signal_flux, signal_time[-1])
np.testing.assert_allclose([sampled_mass, continuous_mass], extra_carbon_mol, rtol=1e-12)
for case in (control, forced):
    run_model(case)
added = integrated_signal(signal_time, signal_flux, forced.time)
print('Control budget:', audit(control))
print('Forced budget:', audit(forced, added))
np.testing.assert_allclose(forced.CO2_At.c[-1] * 1e6, 280, atol=0.5)
np.testing.assert_allclose(forced.Surface.DIC.c[-1] * 1e6, 2040, atol=0.2)
np.testing.assert_allclose(forced.Deep.DIC.c[-1] * 1e6, 2250, atol=0.2)
plot_synthetic_forcing(signal_time, signal_flux, forced, control)
print('Final departure from 280 ppm:', forced.CO2_At.c[-1] * 1e6 - 280)

### What did the forcing experiment establish?

Explain why 62.4 and the mass-based initial inventory lead to the carbon amount
you used. Why should this implementation check return close to 280 ppm? If it
does not, inspect signal units, its integrated mass, carbon/TA conservation,
equilibration and chemistry settings before changing any model inputs.

> **Your explanation:** replace this placeholder with your answer.

The native solver may flag pH changes between stored output points during these
large transients. Use the explicit inventories, stationary flux balance and
endpoint checks above; transient pH interpretation is outside this practical.

**Finish 02:** submit your two derivations (including units) and one sentence
distinguishing the fitted DIC ratio from the conditional atmospheric response.
Point to one internal arrow and the external forcing arrow in your code.